## Mail Sender

Wysyła maila co 10 sekund, od 7:00 do 18:00, od poniedziałku do piątku. Emaile bierze z bazy danych, zapamiętuje na jakie emaile już został wysłany mail, i po przerwaniu pracy zaczyna wysyłać kolejne maile od miejsca gdzie skończył.

In [ ]:
import os
import pandas as pd
from datetime import datetime
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email.mime.image import MIMEImage
from email.mime.application import MIMEApplication
from email import encoders
import base64
import time
import csv

In [ ]:
SCOPES = ['https://www.googleapis.com/auth/gmail.send']

In [ ]:
# Initialization and authorization of Gmail client
def authorize():
    credentials = None
    # Checking if a token exists (generated after the first login)
    if os.path.exists('token.json'):
        credentials = Credentials.from_authorized_user_file('token.json', SCOPES)
    # Authorization if the token is invalid or does not exist
    if not credentials or not credentials.valid:
        if credentials and credentials.expired and credentials.refresh_token:
            credentials.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            credentials = flow.run_local_server(port=8080)
        # Saving token for future use
        with open('token.json', 'w') as token:
            token.write(credentials.to_json())
    return credentials


# Function to send an email with an attachment and image
def send_email(to, subject, message_content, pdf_path=None, image_path=None):
    credentials = authorize()
    service = build('gmail', 'v1', credentials=credentials)
    
    # Set up the MIME message
    message = MIMEMultipart()
    message['From'] = "me"
    message['To'] = to
    message['Subject'] = subject
    
    # Adding HTML content
    html_content = MIMEText(message_content, 'html')
    message.attach(html_content)
    
    # Adding an image as an inline attachment
    if image_path:
        with open(image_path, 'rb') as f:
            img = MIMEImage(f.read())
            img.add_header('Content-ID', '<logo>')  # Setting the cid:logo identifier
            img.add_header('Content-Disposition', 'inline', filename=os.path.basename(image_path))
            message.attach(img)

    # Adding PDF as an attachment
    if pdf_path:
        with open(pdf_path, 'rb') as attachment:
            mime_base = MIMEBase('application', 'pdf')
            mime_base.set_payload(attachment.read())
            encoders.encode_base64(mime_base)
            mime_base.add_header('Content-Disposition', f'attachment; filename="{os.path.basename(pdf_path)}"')
            message.attach(mime_base)
    
    # Encode message and prepare it for sending
    raw_message = base64.urlsafe_b64encode(message.as_bytes()).decode()
    body = {'raw': raw_message}
    
    # Sending message via Gmail API
    try:
        sent_message = service.users().messages().send(userId='me', body=body).execute()
        print(f"E-mail sent to {to} with message ID: {sent_message['id']}")
    except Exception as error:
        print(f"An error occurred: {error}")


def time_to_send():
    now = datetime.now()
    return now.weekday() < 5 and 7 <= now.hour < 18

def load_sent():
    if os.path.exists('sent_emails.csv'):
        with open('sent_emails.csv', mode='r') as file:
            sent = {line.strip() for line in file}
    else:
        sent = set()
    return sent

def save_sent(address):
    with open('sent_emails.csv', mode='a') as file:
        file.write(f"{address}\n")

def save_progress(last_address):
    with open('sending_status.txt', 'w') as file:
        file.write(last_address)

def load_progress():
    if os.path.exists('sending_status.txt'):
        with open('sending_status.txt', 'r') as file:
            return file.read().strip()
    return None

def send_emails_from_database(excel_path, subject, content, pdf_path=None, image_path=None):
    df = pd.read_excel(excel_path)
    email_addresses = df['E-mail'].dropna().tolist()
    
    sent = load_sent()
    last_address = load_progress()
    start = not last_address  # Start true if there is no last address

    for address in email_addresses:
        if address == last_address:  # Found the address from which to continue
            start = True
        if not start or address in sent:  # Skip already sent addresses
            continue

        if time_to_send():
            try:
                # Adding logo to the content of the email
                content_with_logo = content.replace('cid:logo.png', 'cid:logo')
                send_email(address, subject, content_with_logo, pdf_path, image_path)
                save_sent(address)
                save_progress(address)
                print(f"E-mail sent to: {address}")
                time.sleep(10)
            except Exception as e:
                print(f"Failed to send email to {address}: {e}")
                save_progress(address)  # Save progress if error occurs
        else:
            print("Outside working hours. Sending paused.")
            save_progress(address)  # Save the last sent address
            break

In [ ]:
# przyklad uzycia
send_emails_from_database(
    r'C:\Users\User\Jupyter_praca\MailSender\MaileEnyrun.xlsx',
    'Enyrun - przedstawienie katologu produktów',
    '''
    <p>Szanowni Państwo,</p>
    
    <p>Z przyjemnością prezentujemy nasz najnowszy katalog produktów, stworzony z myślą o Państwa potrzebach. Znajdziecie w nim szeroką gamę naszych wyrobów – od sprawdzonych bestsellerów po najnowsze propozycje, które doskonale sprawdzą się w Państwa firmie lub zakładzie.</p>
    
    <p><strong>Dlaczego warto wybrać Enyrun?</strong></p>
    
    <ul>
        <li><strong>Innowacyjność:</strong> Nasza oferta stale się rozwija.</li>
        <li><strong>Jakość:</strong> Wysoka jakość w przystępnej cenie.</li>
        <li><strong>Wzorowa obsługa:</strong> Jesteśmy dostępni telefonicznie i online.</li>
        <li><strong>Sprawna dostawa:</strong> Kurier InPost i DHL na wyciągnięcie ręki.</li>
    </ul>
    
    <p>Zapraszamy do pobrania katalogu i odkrycia, jak nasze produkty mogą ułatwić Państwa codzienną pracę. W przypadku pytań lub potrzeby dodatkowych informacji – jesteśmy do Państwa dyspozycji!</p>
    
    <div style="display: flex; align-items: center; justify-content: space-between;">
        <div>
            <p>Z pozdrowieniami,<br>
            <strong>Zespół Enyrun</strong><br>
            <a href="https://enyrun.pl">enyrun.pl</a><br>
            <a href="mailto:biuro@enyrun.com">biuro@enyrun.com</a><br>
            61 651 78 00<br>
            61 651 78 01</p>
        </div>
        <div style="float: right; text-align: right; margin-top: 2em;">
            <img src="cid:logo.png" width="150" height="75" alt="Logo firmy" style="margin-left: 20px;">
        </div>
    </div>

    <p> Wiadomość wygenerowana automatycznie. Prosimy nie odpowiadać.</p>
    ''',
    r'C:\Users\User\Jupyter_praca\MailSender\Katalog_Enyrun.pdf',
    image_path = r'C:\Users\User\Jupyter_praca\MailSender\logo.png'  
)